# ElectInfo: MOE-aware aggregation + reference crosswalks + cross-tab primitive

## What this shows
Three small statistical primitives ElectInfo relies on when working with ACS data and survey respondents: Margin-of-Error (MOE) propagation for reliable ACS aggregates, NAICS / SOC crosswalks for occupation / industry classification, and the `contingency_table` + `chi_square_test` primitive the survey pipeline delegates to.

## Why it matters
Higher-level pipelines (the survey `WaveSet`, Census-joined reports) all sit on these primitives. Using them directly is the right tool when you're not inside a full survey workflow — e.g. "is the Hispanic share in these tracts reliable enough to report?" or "classify ElectInfo's donor occupations into NAICS sectors."

## Prereqs
- `pip install 'siege-utilities[stats]'`
- No credentials.

## Next
- `reports/03_polling_survey_analysis.ipynb` wraps the cross-tab primitive inside the full survey pipeline.
- `spatial/01_boundaries.ipynb` is where you'd get ACS data in the first place.


## Part 1: Margin of Error Propagation

ACS estimates come with margins of error (MOE) at 90% confidence.
When you combine estimates (sum, subtract, divide), the MOE must be
propagated correctly — you can't just add them.

In [1]:
from siege_utilities.data.statistics.moe_propagation import (
    Estimate,
    moe_sum,
    moe_difference,
    moe_proportion,
    moe_ratio,
    moe_product,
    flag_unreliable,
)

[siege_utilities] 2026-04-24 17:20:30,419 INFO: Initialized Census data source with 23 available years


### 1.1 The Estimate Object

In [2]:
# An ACS estimate with its margin of error
pop_tract_a = Estimate(value=5200, moe=450)
pop_tract_b = Estimate(value=3800, moe=380)
pop_tract_c = Estimate(value=150, moe=120)  # Small population — high CV

for name, est in [("Tract A", pop_tract_a), ("Tract B", pop_tract_b), ("Tract C", pop_tract_c)]:
    lo, hi = est.confidence_interval()
    print(f"{name}: {est.value:,} +/- {est.moe} (CV={est.cv_pct:.1f}%, "
          f"90% CI: [{lo:,.0f}, {hi:,.0f}], reliable={est.is_reliable()})")

Tract A: 5,200 +/- 450 (CV=5.3%, 90% CI: [4,750, 5,650], reliable=True)
Tract B: 3,800 +/- 380 (CV=6.1%, 90% CI: [3,420, 4,180], reliable=True)
Tract C: 150 +/- 120 (CV=48.6%, 90% CI: [30, 270], reliable=False)


### 1.2 Summing Estimates

In [3]:
# Combine tracts into a larger geography
combined = moe_sum([pop_tract_a, pop_tract_b, pop_tract_c])
print(f"Combined: {combined.value:,} +/- {combined.moe:.0f}")
print(f"CV: {combined.cv_pct:.1f}% (improved by aggregation)")
print(f"\nNote: MOE({combined.moe:.0f}) < simple sum({pop_tract_a.moe + pop_tract_b.moe + pop_tract_c.moe})")
print("MOE propagation uses sqrt(sum of squares), not simple addition.")

Combined: 9,150 +/- 601
CV: 4.0% (improved by aggregation)

Note: MOE(601) < simple sum(950)
MOE propagation uses sqrt(sum of squares), not simple addition.


### 1.3 Differences and Proportions

In [4]:
# Difference: how many more people in Tract A than B?
diff = moe_difference(pop_tract_a, pop_tract_b)
print(f"Difference (A - B): {diff.value:,} +/- {diff.moe:.0f}")
lo, hi = diff.confidence_interval()
print(f"90% CI: [{lo:,.0f}, {hi:,.0f}]")
print(f"Statistically significant? {'Yes' if lo > 0 or hi < 0 else 'No (CI includes 0)'}")

print()

# Proportion: what fraction of the combined area is in Tract A?
prop = moe_proportion(pop_tract_a, combined)
print(f"Proportion (A / total): {prop.value:.3f} +/- {prop.moe:.3f}")
print(f"  = {prop.value * 100:.1f}% +/- {prop.moe * 100:.1f} percentage points")

Difference (A - B): 1,400 +/- 589
90% CI: [811, 1,989]
Statistically significant? Yes

Proportion (A / total): 0.568 +/- 0.032
  = 56.8% +/- 3.2 percentage points


### 1.4 Flagging Unreliable Estimates

In [5]:
# Census Bureau considers CV > 40% unreliable
estimates = [pop_tract_a, pop_tract_b, pop_tract_c]
unreliable = flag_unreliable(estimates, cv_threshold=0.40)

print(f"Unreliable estimates (CV > 40%): {len(unreliable)} of {len(estimates)}")
for idx, est in unreliable:
    print(f"  Index {idx}: {est.value:,} +/- {est.moe} (CV={est.cv_pct:.1f}%)")

Unreliable estimates (CV > 40%): 1 of 3
  Index 2: 150 +/- 120 (CV=48.6%)


## Part 2: NAICS/SOC Classification Crosswalks

Federal data uses two classification systems:
- **NAICS** (North American Industry Classification System) — classifies businesses
- **SOC** (Standard Occupational Classification) — classifies workers

siege_utilities provides parsing, hierarchy traversal, and fuzzy matching.

In [6]:
from siege_utilities.reference.naics_soc_crosswalk import (
    parse_naics,
    naics_ancestors,
    naics_to_sector,
    parse_soc,
    soc_to_major_group,
    fuzzy_match_naics,
    NAICS_SECTORS,
    SOC_MAJOR_GROUPS,
)

### 2.1 NAICS Code Parsing

In [7]:
# Parse a NAICS code
code = parse_naics("541511")
print(f"Code: {code.code}")
print(f"Title: {code.title}")
print(f"Level: {code.level} (6-digit = national industry)")
print(f"Sector: {code.sector}")

# Walk up the hierarchy
print(f"\nAncestors of {code.code}:")
for ancestor in naics_ancestors(code.code):
    a = parse_naics(ancestor)
    print(f"  {a.code}: {a.title} (level {a.level})")

Code: 541511
Title: Professional, Scientific, and Technical Services
Level: 6 (6-digit = national industry)
Sector: 54

Ancestors of 541511:
  54: Professional, Scientific, and Technical Services (level 2)
  541: Professional, Scientific, and Technical Services (level 3)
  5415: Professional, Scientific, and Technical Services (level 4)
  54151: Professional, Scientific, and Technical Services (level 5)
  541511: Professional, Scientific, and Technical Services (level 6)


### 2.2 Sector Overview

In [8]:
print(f"NAICS has {len(NAICS_SECTORS)} sectors:")
for code, title in sorted(NAICS_SECTORS.items()):
    print(f"  {code}: {title}")

NAICS has 24 sectors:
  11: Agriculture, Forestry, Fishing and Hunting
  21: Mining, Quarrying, and Oil and Gas Extraction
  22: Utilities
  23: Construction
  31: Manufacturing
  32: Manufacturing
  33: Manufacturing
  42: Wholesale Trade
  44: Retail Trade
  45: Retail Trade
  48: Transportation and Warehousing
  49: Transportation and Warehousing
  51: Information
  52: Finance and Insurance
  53: Real Estate and Rental and Leasing
  54: Professional, Scientific, and Technical Services
  55: Management of Companies and Enterprises
  56: Administrative and Support and Waste Management
  61: Educational Services
  62: Health Care and Social Assistance
  71: Arts, Entertainment, and Recreation
  72: Accommodation and Food Services
  81: Other Services (except Public Administration)
  92: Public Administration


### 2.3 SOC Codes

In [9]:
# Parse an SOC code
soc = parse_soc("15-1252")
print(f"SOC: {soc.code} — {soc.title}")
print(f"Level: {soc.level}")
print(f"Major group: {soc.major_group}")

major_code, major_title = soc_to_major_group("15-1252")
print(f"Major group: {major_code} — {major_title}")

print(f"\nSOC has {len(SOC_MAJOR_GROUPS)} major groups:")
for code, title in sorted(SOC_MAJOR_GROUPS.items())[:5]:
    print(f"  {code}: {title}")
print(f"  ... and {len(SOC_MAJOR_GROUPS) - 5} more")

SOC: 15-1252 — Computer and Mathematical
Level: detailed
Major group: 15
Major group: 15 — Computer and Mathematical

SOC has 23 major groups:
  11: Management
  13: Business and Financial Operations
  15: Computer and Mathematical
  17: Architecture and Engineering
  19: Life, Physical, and Social Science
  ... and 18 more


### 2.4 Fuzzy Matching

In [10]:
# Match free-text to NAICS sectors
for text in ["software consulting", "restaurant food", "construction building", "health care"]:
    matches = fuzzy_match_naics(text, threshold=0.3)
    if matches:
        best = matches[0]
        print(f"  '{text}' → {best[0]}: {best[1]} (score={best[2]:.2f})")
    else:
        print(f"  '{text}' → no match")

  'software consulting' → no match
  'restaurant food' → no match
  'construction building' → 23: Construction (score=0.50)
  'health care' → 62: Health Care and Social Assistance (score=0.40)


## Part 3: Cross-tabulation primitive

`siege_utilities.data.statistics.cross_tabulation` is the primitive the survey
pipeline delegates to. Use it directly for quick counts / percentages without
a full Chain → Argument workflow.


In [11]:
import pandas as pd
from siege_utilities.data.statistics.cross_tabulation import (
    contingency_table, rate_table, chi_square_test,
)

df = pd.DataFrame({
    'party':  ['D', 'R', 'D', 'R', 'I', 'D', 'R', 'D'],
    'county': ['Travis', 'Travis', 'Harris', 'Harris',
               'Harris', 'Dallas', 'Dallas', 'Dallas'],
})

# Counts
ct = contingency_table(df, row_var='party', col_var='county')
print('contingency table:')
print(ct)

# Chi-square test on the table
result = chi_square_test(ct)
print(f'\nchi-square: statistic={result.statistic:.3f}, p={result.p_value:.4f}, dof={result.dof}')


contingency table:
county  Dallas  Harris  Travis  Total
party                                
D            2       1       1      4
I            0       1       0      1
R            1       1       1      3
Total        3       3       2      8



chi-square: statistic=2.111, p=0.7153, dof=4


## Related

- Source: `siege_utilities/data/statistics/` (MOE, cross-tabulation),
  `siege_utilities/reference/` (NAICS/SOC crosswalks)
- Tests: `tests/test_moe.py`, `tests/test_cross_tabulation.py`, `tests/test_naics_soc.py`
- Higher-level wrapper: `siege_utilities/survey/crosstab.py` (used in `reports/03_polling_survey_analysis.ipynb`)
